In [11]:
"""
修正版：避免重复模板
"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from typing import List, Dict, Optional

class SimpleLLM:
    """带上下文的 LLM 对话类（修正版）"""
    
    def __init__(self, model_name: str = "Qwen/Qwen2.5-1.5B-Instruct", max_history: int = 10):
        print(f"🔄 加载模型: {model_name}")
        
        self.max_history = max_history
        self.messages = []  # 只存储原始内容，不存储模板格式
        
        # 检测设备
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"  使用设备: {self.device}")
        
        # 加载 tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        
        # 加载模型
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
        
        # 创建 pipeline（不返回完整文本）
        pipe = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            return_full_text=False  # 重要：不返回输入文本
        )
        
        self.llm = HuggingFacePipeline(pipeline=pipe)
        print("✅ 模型加载完成")
    
    def add_user_message(self, content: str):
        """添加用户消息（只存原始内容）"""
        self.messages.append({"role": "user", "content": content})
        self._trim_history()
    
    def add_assistant_message(self, content: str):
        """添加助手消息（只存原始内容）"""
        self.messages.append({"role": "assistant", "content": content})
        self._trim_history()
    
    def _trim_history(self):
        """修剪历史记录"""
        if len(self.messages) > self.max_history * 2:
            self.messages = self.messages[-(self.max_history * 2):]
    
    def clear_history(self):
        """清空历史"""
        self.messages = []
    
    def set_system_prompt(self, system_prompt: str):
        """设置系统提示词（只存一次）"""
        # 移除旧的系统提示
        self.messages = [m for m in self.messages if m["role"] != "system"]
        # 添加新的系统提示到开头
        self.messages.insert(0, {"role": "system", "content": system_prompt})
    
    def chat(self, user_input: str) -> str:
        """
        带上下文的对话（修正版）
        
        Args:
            user_input: 用户输入
            
        Returns:
            AI 回复（纯文本，不含模板）
        """
        # 1. 准备消息列表（只包含原始内容）
        messages = self.messages.copy()
        messages.append({"role": "user", "content": user_input})
        
        # 2. 应用聊天模板（只应用一次）
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # 3. 生成回复
        response = self.llm.invoke(prompt)
        
        # 4. 清理回复（去除可能的特殊标记）
        response = self._clean_response(response)
        
        # 5. 保存到历史
        self.add_user_message(user_input)
        self.add_assistant_message(response)
        
        return response
    
    def _clean_response(self, response: str) -> str:
        """清理回复中的特殊标记"""
        # 去除可能的模板标记
        markers = ["<|im_start|>", "<|im_end|>", "<|endoftext|>"]
        for marker in markers:
            response = response.replace(marker, "")
        
        # 去除多余的空行
        lines = [line.strip() for line in response.split('\n') if line.strip()]
        response = '\n'.join(lines)
        
        return response
    
    def get_history(self) -> List[Dict]:
        """获取对话历史"""
        return self.messages.copy()


In [12]:
llm = SimpleLLM(max_history=5)

🔄 加载模型: Qwen/Qwen2.5-1.5B-Instruct
  使用设备: cuda


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ 模型加载完成


In [14]:
# 使用示例
def play():
    
    # 设置系统提示词
    system_prompt = "你是一个友好的餐厅客服，名叫小美。"
    llm.set_system_prompt(system_prompt)
    
    # 多轮对话
    response1 = llm.chat("你好，我叫张三")
    print(f"AI: {response1}")
    
    response2 = llm.chat("我叫什么名字？")
    print(f"AI: {response2}")  # ✅ 现在能记住名字了！
    
    response3 = llm.chat("刚才我们聊了什么？")
    print(f"AI: {response3}")  # ✅ 能回顾之前的对话！
    
    # 查看历史
    #print("\n📋 对话历史:")
    #for msg in llm.get_history():
     #   print(f"  {msg['role']}: {msg['content'][:30]}...")

play()    

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI: 您好，张先生！很高兴为您服务，请问有什么可以帮到您的呢？


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AI: 您是张先生吗？那么恭喜您成为我们的新会员，欢迎光临我们餐厅！
AI: 我们在聊天中确认了我的身份，并且祝贺您成为我们的新会员。如果您有任何问题或需要帮助，请随时告诉我。


In [8]:
llm.chat("你是谁")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n你好，我叫张三<|im_end|>\n<|im_start|>assistant\n<|im_start|>system\n你是一个友好的餐厅客服，名叫小美。<|im_end|>\n<|im_start|>user\n你好，我叫张三<|im_end|>\n<|im_start|>assistant\n您好！张先生，欢迎来到我们的餐厅。有什么我可以帮您的吗？<|im_end|>\n<|im_start|>user\n我叫什么名字？<|im_end|>\n<|im_start|>assistant\n<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n你好，我叫张三<|im_end|>\n<|im_start|>assistant\n<|im_start|>system\n你是一个友好的餐厅客服，名叫小美。<|im_end|>\n<|im_start|>user\n你好，我叫张三<|im_end|>\n<|im_start|>assistant\n您好！张先生，欢迎来到我们的餐厅。有什么我可以帮您的吗？<|im_end|>\n<|im_start|>user\n我叫什么名字？<|im_end|>\n<|im_start|>assistant\n您是张先生，对吧？很高兴见到您！有什么我可以帮助您的吗？<|im_end|>\n<|im_start|>user\n刚才我们聊了什么？<|im_end|>\n<|im_start|>assistant\n<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n你好，我叫张三<|im_end|>\n<|im_start|>assistant\